# Circular Pad — Curved Geometry and Palace Eigenmode

This synthetic single-die LC model uses OrPen’s
`circular_pad_capacitor`: radius 100 µm, radial gap 20 µm, 1000 µm die,
Al thickness 0.2 µm and Si thickness 500 µm. It is an electrical
electrode, not a mechanical drum or measured qubit. Component source
owns the local PAD/GROUND Entities, positive Ground polygon with its
opening, exact circular boundary chains and east-facing non-metal
junction locator. The Notebook owns final electrical Nets and the linear
10 nH branch; C=0 denotes no added capacitive branch.

The 2 µm-wide sheet lands on the curved pad over its full width. Its
left edge is `sqrt(R²-(width/2)²)-1 DBU`, not `R-1 DBU`; Ground landing
extends to `R+gap+1 DBU`. This authored overlap is not an accuracy
tolerance or an instruction to infer connectivity. The compiler resolves
actual intersections.

Canonical source is this QMD; IPYNB is derived and documentation never
executes it. This CONVERGING workflow requires SCGSim’s explicit
curve/mesh API. The project pins the obtainable SCGSim develop revision
`cd107e4b0d2cca48ef9d58143e0b4b3d3271a263` (`1.1.0rc1`). The initial
preparation used a separately sealed internal wheel with matching
runtime bytes; it is not an observation of a stable release.
Preparation, execution and returned results are separate observations.
No convergence, frequency, cross-solver or accuracy Gate applies.

The assigned native observation prepared the authored second-order mesh
and handoff, but the selected Palace build stopped during lumped-port
geometry verification before an eigenmode or AMR pass was produced. No
frequency or EPR result is available for that attempt; failed receipts
remain failed. This CONVERGING notebook does not imply a completed
circular-LC solver example.

In [ ]:
from pathlib import Path
import os
import subprocess
import importlib.metadata

import gdsfactory as gf
from IPython.display import display
from scgsim.sgb import GeometryPlan
from scgsim.palace import EigenmodeSim, resolve_palace_result, inspect_run_trustworthiness
import orpen_sc_pdk
from orpen_sc_pdk import get_material_records, get_interface_preset_records, validate_interface_preset_records
from orpen_sc_pdk.tech import get_single_die_layer_stack

WORKFLOW_ACTION = "prepare_handoff"  # prepare_handoff | run | analyze_handoff
RUN_ID = "circular_pad_palace_eigenmode_20261007_04"
OUTPUT_ROOT = Path.cwd() / ".artifacts" / "CircularPad"
RUN_ROOT = OUTPUT_ROOT / RUN_ID
if WORKFLOW_ACTION not in {"prepare_handoff", "run", "analyze_handoff"}:
    raise ValueError("Unknown WORKFLOW_ACTION")
display({"scgsim_version": importlib.metadata.version("scgsim"), "run_id": RUN_ID})

## Build Component Coupon

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    OUTER_RADIUS_UM = 100.0
    INNER_RADIUS_UM = 0.0
    GAP_UM = 20.0
    DIE_SIDE_UM = 1000.0
    JUNCTION_WIDTH_UM = 2.0
    orpen_sc_pdk.activate()
    gf.clear_cache()
    component = gf.get_component(
        "circular_pad_capacitor", outer_radius_um=OUTER_RADIUS_UM,
        inner_radius_um=INNER_RADIUS_UM, gap_um=GAP_UM,
        die_side_um=DIE_SIDE_UM, junction_width_um=JUNCTION_WIDTH_UM,
    )
    layer_stack = get_single_die_layer_stack()
    plan = GeometryPlan(
        component=component, root_id="chip", layer_stack=layer_stack,
        material_records=get_material_records(), coupon_padding_um=0.0,
    )
    plan.set_nets({"pad": ("chip/PAD",), "ground": ("chip/GROUND",)})
    plan.set_vacuum_region(padding=(250.0, 250.0, 1000.0))
    snapshot = plan.prepare()
    display(component)
    display({"bbox_um": str(component.dbbox()), "ports": component.ports,
             "paired_gds_sha256": snapshot.gds_sha256})

## Configure EPR / Problem

LiteratureCentral MA/MS/SA are synthetic model conventions, not measured
process data. EPR is baseline unmasked / zero-inset only. Positive-inset
perimeter distances use chords in the selected Palace runtime; this
workflow does not claim exact curved mask separation or introduce a mask
study.

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    ROUTE = "B"
    sim = EigenmodeSim()
    sim.set_plan(snapshot)
    sim.set_output_dir(RUN_ROOT)
    records = validate_interface_preset_records(get_interface_preset_records())
    preset_names = {kind: f"LiteratureCentral_{kind}" for kind in ("MA", "MS", "SA")}
    interface_presets = {kind: records[name] for kind, name in preset_names.items()}
    sim.set_surface_epr(
        representation=ROUTE,
        specs={kind: {
            "thickness": record["thickness"], "permittivity": record["permittivity"],
            "loss_tangent": record["loss_tangent"], "source": record["source"],
            "preset": preset_names[kind],
        } for kind, record in interface_presets.items()},
    )
    PORT_NAME = "chip/o_junction_lumped"
    PORT_INDUCTANCE_H = 10e-9
    PORT_CAPACITANCE_F = 0.0  # Inductive-only model; no Palace capacitive branch.
    sim.add_port(PORT_NAME, layer="D0_TOP_M1", layout_sheet=True, inductance=PORT_INDUCTANCE_H)
    sim.set_epr_request(surface_interfaces=("MA", "MS", "SA"), bulk_domain_ids=None,
                        port_names=(PORT_NAME,))
    NUM_MODES = 1
    SEARCH_FREQUENCY_HZ = 1e9  # Search lower bound, not a desired frequency.
    EIGENMODE_TOLERANCE = 1e-6
    SAVE_FIELDS = 1
    sim.set_eigenmode(num_modes=NUM_MODES, target=SEARCH_FREQUENCY_HZ,
                      tolerance=EIGENMODE_TOLERANCE, save=SAVE_FIELDS)
    display({"route": ROUTE, "inductance_h": PORT_INDUCTANCE_H,
             "capacitance_f": PORT_CAPACITANCE_F, "interface_presets": interface_presets})

## Build Mesh

Geometric order is independent of field FEM order. SCGSim owns the
shared curve arrangement, source rebinding, elevation and one HighOrder
optimization.

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    REFINED_MESH_SIZE_UM = 10.0
    MAX_MESH_SIZE_UM = 200.0
    ALGORITHM_3D = "HXT"
    MESH_THREADS = 8
    SURFACE_THREADS = 1
    GEOMETRY_ORDER = 2
    HIGH_ORDER_OPTIMIZE = True
    sim.set_mesh(refined_mesh_size=REFINED_MESH_SIZE_UM, max_mesh_size=MAX_MESH_SIZE_UM,
                 algorithm_3d=ALGORITHM_3D, threads=MESH_THREADS,
                 surface_threads=SURFACE_THREADS, geometry_order=GEOMETRY_ORDER,
                 high_order_optimize=HIGH_ORDER_OPTIMIZE)
    MESH_PATH = sim.mesh()
    display(MESH_PATH)
    display(sim.mesh_summary())

## Generate Config

The maximum permits native early stopping; four completed AMR passes are
not required. Controls are declared inputs, not scientific acceptance
criteria.

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    FEM_ORDER = 2
    LINEAR_TOLERANCE = 1e-6
    MAX_ITERATIONS = 2000
    AMR_MAX_PASSES = 4
    AMR_TOLERANCE = 0.02
    AMR_UPDATE_FRACTION = 0.15
    AMR_NONCONFORMAL = False
    SAVE_ADAPT_ITERATIONS = True
    sim.set_numerical(
        order=FEM_ORDER, tolerance=LINEAR_TOLERANCE, max_iterations=MAX_ITERATIONS,
        solver_type="Default", preconditioner="Default", device="CPU",
        amr_max_passes=AMR_MAX_PASSES, amr_tolerance=AMR_TOLERANCE,
        amr_update_fraction=AMR_UPDATE_FRACTION, amr_nonconformal=AMR_NONCONFORMAL,
        save_adapt_iterations=SAVE_ADAPT_ITERATIONS,
        output_paraview=True, output_grid_function=False,
    )
    CONFIG_PATH = sim.write_config()
    display(CONFIG_PATH)

## Prepare Handoff

Only `run` invokes the generated script. Use a fresh RUN_ID for each new
preparation; existing destinations fail rather than being overwritten.
Exact host executable/setup provenance belongs in the owner’s execution
receipt, not a machine-local public default. There is no automatic retry
or fallback. `PALACE_EXECUTABLE` must select an actual Palace MPI
wrapper: the generated script supplies `-np 8 -nt 1`. A direct ELF is
not a wrapper.

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    MACHINE_PROFILE = "direct-local"
    PALACE_EXECUTABLE = os.environ.get("PALACE_EXECUTABLE", "palace")
    SETUP_COMMANDS = tuple(os.environ.get("PALACE_SETUP_COMMANDS", "").splitlines())
    RESOURCES = {"processes": 8, "threads": 1, "command_style": "wrapper"}
    HANDOFF = sim.prepare_handoff(profile=MACHINE_PROFILE, executable=PALACE_EXECUTABLE,
                                  resources=RESOURCES, setup_commands=SETUP_COMMANDS)
    display(HANDOFF)
    display({"retain_independently": HANDOFF.handoff_id})
    if WORKFLOW_ACTION == "run":
        subprocess.run([str(HANDOFF.script_path)], cwd=HANDOFF.run_dir, check=True)

## Analyze Returned Run

Supply the existing returned directory and independently retained
handoff ID in a fresh analysis kernel. Failed/partial results stay
failed/partial; they never become a strict completed result. Analyze
does not build or solve.

### Run Identity

### Numerical Evidence

### Simulation Benchmark

In [ ]:
if WORKFLOW_ACTION == "analyze_handoff":
    RETURNED_RUN_DIR = Path(os.environ["CIRCULAR_PAD_RETURNED_RUN_DIR"])
    EXPECTED_HANDOFF_ID = os.environ["CIRCULAR_PAD_PALACE_HANDOFF_ID"]
    report = inspect_run_trustworthiness(RETURNED_RUN_DIR)
    display(report)
    if report.completeness == "complete":
        result = resolve_palace_result(RETURNED_RUN_DIR, expected_handoff_id=EXPECTED_HANDOFF_ID)
        display(result.show_run_trustworthiness())

### Physics Analysis Results

In [ ]:
if WORKFLOW_ACTION == "analyze_handoff":
    if report.completeness == "complete":
        result.show_all_results()
    else:
        report.show_all_results()

### Simulation Performance / Benchmarks

Read actual emitted geometric node/order data and native AMR histories
beside the declared controls. Imported p2 support in source does not
establish node preservation after port processing or refinement. Missing
channels and native failures remain unavailable or failed; no accuracy,
convergence, lifetime or high-order-preservation success is inferred
from execution alone.